# Введение в нейронные сети

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
from matplotlib.colors import LinearSegmentedColormap
from sklearn.datasets import make_moons
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from torch import nn, optim

In [ ]:
RANDOM_STATE = 42
N_SAMPLES = 3000
EPOCHS = 100
LEARNING_RATE = 0.1
INPUT_SIZE = 2
HIDDEN_SIZE = 8
OUTPUT_SIZE = 1

np.random.seed(RANDOM_STATE)
torch.manual_seed(RANDOM_STATE)

COLORS = ("#4477AA", "#EE7733")

plt.rcParams.update(
    {
        "figure.figsize": (8, 5),
        "axes.spines.top": False,
        "axes.spines.right": False,
        "axes.grid": True,
        "grid.alpha": 0.2,
        "font.size": 11
    }
)

## Создание и визуализация датасета

In [ ]:
x, y = make_moons(n_samples=N_SAMPLES, noise=0.125, random_state=RANDOM_STATE)

x_train_valid, x_test, y_train_valid, y_test = train_test_split(
    x, y, test_size=0.2, stratify=y, random_state=RANDOM_STATE
)
x_train, x_valid, y_train, y_valid = train_test_split(
    x_train_valid,
    y_train_valid,
    test_size=0.25,
    stratify=y_train_valid,
    random_state=RANDOM_STATE,
)

scaler = StandardScaler()
x_train_scaled = scaler.fit_transform(x_train)
x_valid_scaled = scaler.transform(x_valid)
x_test_scaled = scaler.transform(x_test)

split_sizes = pd.DataFrame(
    {
        "Выборка": ["Train", "Validation", "Test"],
        "Число объектов": [len(y_train), len(y_valid), len(y_test)],
        "Доля класса 1": [y_train.mean(), y_valid.mean(), y_test.mean()],
    }
).set_index("Выборка")
display(split_sizes.round(3))

In [ ]:
fig, ax = plt.subplots()
for class_id, color in enumerate(COLORS):
    mask = y_train == class_id
    ax.scatter(
        x_train_scaled[mask, 0],
        x_train_scaled[mask, 1],
        color=color,
        s=18,
        alpha=0.6,
        label=f"Класс {class_id}",
    )
ax.set(
    title="Обучающая выборка",
    xlabel="Признак 1 после стандартизации",
    ylabel="Признак 2 после стандартизации",
)
ax.legend()
fig.tight_layout()
plt.show()

#### Архитектура сети:
$$
\begin{align*}
z_1 &= X \cdot W_1 + b_1 \\
a_1 &= \text{ReLU}(z_1) \\
z_2 &= a_1 \cdot W_2 + b_2 \\
a_2 &= \sigma(z_2)
\end{align*}
$$

#### Функции активации:
$$
\begin{align*}
\text{ReLU}(x) &= \max(0, x) \\
\sigma(x) &= \frac{1}{1 + e^{-x}}
\end{align*}
$$

#### Функция потерь (Binary Cross-Entropy):
$$
L = -\frac{1}{m} \sum_{i=1}^{m} \left[ y_i \cdot \log(\hat{y}_i) + (1 - y_i) \cdot \log(1 - \hat{y}_i) \right]
$$

In [ ]:
class NumpyMLP:
    """Нейронная сеть для бинарной классификации с одним скрытым слоем."""

    def __init__(
        self,
        input_size: int,
        hidden_size: int,
        output_size: int,
        learning_rate: float,
    ) -> None:
        # Инициализация Kaiming для слоя с ReLU.
        self.w1 = np.random.randn(input_size, hidden_size) * np.sqrt(2 / input_size)
        self.b1 = np.zeros((1, hidden_size))
        # Инициализация Xavier для выходного слоя.
        self.w2 = np.random.randn(hidden_size, output_size) * np.sqrt(
            2 / (hidden_size + output_size)
        )
        self.b2 = np.zeros((1, output_size))
        self.learning_rate = learning_rate

    @staticmethod
    def sigmoid(z: np.ndarray) -> np.ndarray:
        # Ограничиваем аргумент экспоненты, чтобы избежать переполнения.
        return 1 / (1 + np.exp(-np.clip(z, -500, 500)))

    @staticmethod
    def binary_cross_entropy(y_true: np.ndarray, y_pred: np.ndarray) -> float:
        y_true = y_true.reshape(-1, 1)
        probabilities = np.clip(y_pred, 1e-12, 1 - 1e-12)
        loss = -np.mean(
            y_true * np.log(probabilities) + (1 - y_true) * np.log(1 - probabilities)
        )
        return float(loss)

    def forward(self, x: np.ndarray) -> np.ndarray:
        self.z1 = x @ self.w1 + self.b1
        self.a1 = np.maximum(0, self.z1)
        self.z2 = self.a1 @ self.w2 + self.b2
        return self.sigmoid(self.z2)

    def backward(self, x: np.ndarray, y_true: np.ndarray, y_pred: np.ndarray) -> None:
        sample_count = len(x)
        dz2 = y_pred - y_true.reshape(-1, 1)
        dw2 = self.a1.T @ dz2 / sample_count
        db2 = dz2.mean(axis=0, keepdims=True)

        dz1 = (dz2 @ self.w2.T) * (self.z1 > 0)
        dw1 = x.T @ dz1 / sample_count
        db1 = dz1.mean(axis=0, keepdims=True)

        self.w1 -= self.learning_rate * dw1
        self.b1 -= self.learning_rate * db1
        self.w2 -= self.learning_rate * dw2
        self.b2 -= self.learning_rate * db2

    def predict(self, x: np.ndarray) -> np.ndarray:
        return (self.forward(x).ravel() >= 0.5).astype(int)

### Обучение

В каждой эпохе передаём в сеть **всю обучающую выборку** и один раз обновляем веса. Это полный градиентный спуск.

In [ ]:
model_numpy = NumpyMLP(
    input_size=INPUT_SIZE,
    hidden_size=HIDDEN_SIZE,
    output_size=OUTPUT_SIZE,
    learning_rate=LEARNING_RATE,
)
history_numpy = {
    "train_loss": [],
    "valid_loss": [],
    "train_accuracy": [],
    "valid_accuracy": []
}

In [ ]:
for epoch in range(1, EPOCHS + 1):
    # прямой проход
    train_probabilities = model_numpy.forward(x_train_scaled)
    # обратный проход
    model_numpy.backward(x_train_scaled, y_train, train_probabilities)

    train_probabilities = model_numpy.forward(x_train_scaled)
    valid_probabilities = model_numpy.forward(x_valid_scaled)

    # вычисление значения функции потерь
    train_loss = model_numpy.binary_cross_entropy(y_train, train_probabilities)
    valid_loss = model_numpy.binary_cross_entropy(y_valid, valid_probabilities)

    # оценка метрики
    train_accuracy = np.mean((train_probabilities.ravel() >= 0.5) == y_train)
    valid_accuracy = np.mean((valid_probabilities.ravel() >= 0.5) == y_valid)

    # сохранение истории
    history_numpy["train_loss"].append(train_loss)
    history_numpy["valid_loss"].append(valid_loss)
    history_numpy["train_accuracy"].append(float(train_accuracy))
    history_numpy["valid_accuracy"].append(float(valid_accuracy))

    if epoch == 1 or epoch % 250 == 0 or epoch == EPOCHS:
        print(
            f"Эпоха {epoch:4d} | Loss: train={train_loss:.4f}, valid={valid_loss:.4f} | "
            f"Accuracy: train={train_accuracy:.4f}, valid={valid_accuracy:.4f}"
        )

#### Визуализация процесса обучения НС
##### Кривые обучения

Посмотрим, как меняются ошибка и доля правильных ответов.
Если train loss продолжает снижаться, а validation loss устойчиво растёт,
это признак переобучения. Accuracy при этом может почти не меняться:
она учитывает только класс после применения порога, а не саму вероятность.

In [ ]:
def plot_learning_history(history: dict[str, list[float]], title: str) -> None:
    """
    Показать loss и accuracy на train и validation
    """
    epochs = np.arange(1, len(history["train_loss"]) + 1)
    fig, axes = plt.subplots(1, 2, figsize=(12, 4))
    for split, label, color in [
        ("train", "Train", COLORS[0]),
        ("valid", "Validation", COLORS[1]),
    ]:
        axes[0].plot(epochs, history[f"{split}_loss"], label=label, color=color)
        axes[1].plot(epochs, history[f"{split}_accuracy"], label=label, color=color)
    axes[0].set(title="Функция потерь", xlabel="Эпоха", ylabel="BCE")
    axes[1].set(
        title="Доля правильных ответов",
        xlabel="Эпоха",
        ylabel="Accuracy",
        ylim=(0, 1.02),
    )
    for ax in axes:
        ax.legend()
    fig.suptitle(title)
    fig.tight_layout()
    plt.show()


plot_learning_history(history_numpy, "Обучение сети на NumPy")

### Карта вероятностей

Цвет фона показывает предсказанную вероятность класса 1.

Чёрная линия — граница $p=0{,}5$.

Точки — объекты validation с известными классами.


In [ ]:
grid_x, grid_y = np.meshgrid(
    np.linspace(
        x_train_scaled[:, 0].min() - 0.3, x_train_scaled[:, 0].max() + 0.3, 250
    ),
    np.linspace(
        x_train_scaled[:, 1].min() - 0.3, x_train_scaled[:, 1].max() + 0.3, 250
    ),
)
grid_points = np.column_stack([grid_x.ravel(), grid_y.ravel()])


def plot_probability_map(
    grid_x: np.ndarray,
    grid_y: np.ndarray,
    probabilities: np.ndarray,
    x: np.ndarray,
    y: np.ndarray,
    title: str
) -> None:
    """Показать вероятности, границу классов и отложенные объекты."""
    probabilities = probabilities.reshape(grid_x.shape)
    fig, ax = plt.subplots(figsize=(8, 5))
    background = ax.contourf(
        grid_x,
        grid_y,
        probabilities,
        levels=np.linspace(0, 1, 21),
        cmap=LinearSegmentedColormap.from_list("classes", COLORS),
        alpha=0.8
    )
    ax.contour(
        grid_x, grid_y, probabilities, levels=[0.5], colors="black", linewidths=1.5
    )
    for class_id, color in enumerate(COLORS):
        mask = y == class_id
        ax.scatter(
            x[mask, 0],
            x[mask, 1],
            color=color,
            edgecolors="white",
            linewidths=0.4,
            s=20,
            label=f"Класс {class_id}"
        )
    fig.colorbar(background, ax=ax, label="Вероятность класса 1")
    ax.set(
        title=title,
        xlabel="Признак 1 после стандартизации",
        ylabel="Признак 2 после стандартизации",
        xlim=(grid_x.min(), grid_x.max()),
        ylim=(grid_y.min(), grid_y.max())
    )
    ax.legend()
    fig.tight_layout()
    plt.show()


numpy_grid_probabilities = model_numpy.forward(grid_points)
plot_probability_map(
    grid_x,
    grid_y,
    numpy_grid_probabilities,
    x_valid_scaled,
    y_valid,
    "Карта вероятностей: NumPy"
)

## Создание НС с использованием PyTorch
Архитектура и функция потерь останутся теми же. Слои `nn.Linear` хранят веса
и смещения, `loss.backward()` вычисляет градиенты, а оптимизатор обновляет параметры.

В обеих версиях используем инициализации Kaiming для первого слоя и Xavier
для второго.

Конкретные начальные веса всё равно будут разными: NumPy и PyTorch
используют разные генераторы случайных чисел.

In [ ]:
class TorchMLP(nn.Module):
    """Та же архитектура с автоматическим вычислением градиентов."""

    def __init__(self, input_size: int, hidden_size: int, output_size: int) -> None:
        super().__init__()
        self.network = nn.Sequential(
            nn.Linear(input_size, hidden_size),
            nn.ReLU(),
            nn.Linear(hidden_size, output_size),
            nn.Sigmoid(),
        )
        nn.init.kaiming_normal_(self.network[0].weight, nonlinearity="relu")
        nn.init.zeros_(self.network[0].bias)
        nn.init.xavier_normal_(self.network[2].weight)
        nn.init.zeros_(self.network[2].bias)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return self.network(x)

In [ ]:
# преобразуем данные в torch tensor тип
x_train_tensor = torch.as_tensor(x_train_scaled, dtype=torch.float32)
y_train_tensor = torch.as_tensor(y_train, dtype=torch.float32).reshape(-1, 1)
x_valid_tensor = torch.as_tensor(x_valid_scaled, dtype=torch.float32)
y_valid_tensor = torch.as_tensor(y_valid, dtype=torch.float32).reshape(-1, 1)

### Обучение и оценка

На каждом шаге:

1. `model.train()` — включаем режим обучения.
2. `optimizer.zero_grad()` — обнуляем градиенты предыдущего шага.
3. Выполняем прямой проход, вычисляем loss и вызываем `loss.backward()`.
4. `optimizer.step()` — обновляем параметры.
5. `model.eval()` и `torch.no_grad()` — оцениваем обновлённую модель без записи графа вычислений.

`eval()` переключает поведение таких слоёв, как Dropout и BatchNorm, но не отключает
градиенты.

За отключение записи графа отвечает `no_grad()`.

В нашей сети нет этих
слоёв, однако сохраняем стандартный порядок работы.



In [ ]:

model_torch = TorchMLP(INPUT_SIZE, HIDDEN_SIZE, OUTPUT_SIZE)

history_torch = {
    "train_loss": [],
    "valid_loss": [],
    "train_accuracy": [],
    "valid_accuracy": []
}

# Задаем Loss и оптимизатор
criterion = nn.BCELoss()
optimizer = optim.SGD(model_torch.parameters(), lr=LEARNING_RATE)

In [ ]:
for epoch in range(1, EPOCHS + 1):
    model_torch.train()
    optimizer.zero_grad() # обнуляем градиенты потому что torch по дефолту накапливает значения градиентов

    # прямой проход
    train_probabilities = model_torch(x_train_tensor)

    # вычисление значения функции потерь
    loss = criterion(train_probabilities, y_train_tensor)
    loss.backward() # вычисляем градиенты
    optimizer.step() # обновляем веса

    model_torch.eval()
    # не используем автоград при расчете метрики (а так же при валидации и инференсе)
    with torch.no_grad():
        train_probabilities = model_torch(x_train_tensor)
        valid_probabilities = model_torch(x_valid_tensor)
        train_loss = criterion(train_probabilities, y_train_tensor).item()
        valid_loss = criterion(valid_probabilities, y_valid_tensor).item()
        train_accuracy = (
            ((train_probabilities >= 0.5) == y_train_tensor).float().mean().item()
        )
        valid_accuracy = (
            ((valid_probabilities >= 0.5) == y_valid_tensor).float().mean().item()
        )

    # сохранение истории
    history_torch["train_loss"].append(train_loss)
    history_torch["valid_loss"].append(valid_loss)
    history_torch["train_accuracy"].append(train_accuracy)
    history_torch["valid_accuracy"].append(valid_accuracy)

    if epoch == 1 or epoch % 250 == 0 or epoch == EPOCHS:
        print(
            f"Эпоха {epoch:4d} | Loss: train={train_loss:.4f}, valid={valid_loss:.4f} | "
            f"Accuracy: train={train_accuracy:.4f}, valid={valid_accuracy:.4f}"
        )

#### Визуализация процесса обучения НС

In [ ]:
plot_learning_history(history_torch, "Обучение сети на PyTorch")

In [ ]:
model_torch.eval()
with torch.no_grad():
    grid_tensor = torch.as_tensor(grid_points, dtype=torch.float32)
    torch_grid_probabilities = model_torch(grid_tensor).numpy()

plot_probability_map(
    grid_x,
    grid_y,
    torch_grid_probabilities,
    x_valid_scaled,
    y_valid,
    "Карта вероятностей: PyTorch"
)

## Сравнение моделей

Теперь один раз оценим обе обученные модели на тестовой выборке.
В этом примере сравниваем модели после одинакового числа эпох.

In [ ]:
numpy_test_probabilities = model_numpy.forward(x_test_scaled)
numpy_test_loss = model_numpy.binary_cross_entropy(y_test, numpy_test_probabilities)
numpy_test_accuracy = np.mean((numpy_test_probabilities.ravel() >= 0.5) == y_test)

model_torch.eval()
with torch.no_grad():
    x_test_tensor = torch.as_tensor(x_test_scaled, dtype=torch.float32)
    y_test_tensor = torch.as_tensor(y_test, dtype=torch.float32).reshape(-1, 1)
    torch_test_probabilities = model_torch(x_test_tensor)
    torch_test_loss = criterion(torch_test_probabilities, y_test_tensor).item()
    torch_test_accuracy = (
        ((torch_test_probabilities >= 0.5) == y_test_tensor).float().mean().item()
    )

test_results = pd.DataFrame(
    {
        "Модель": ["NumPy", "PyTorch"],
        "Test BCE": [numpy_test_loss, torch_test_loss],
        "Test accuracy": [float(numpy_test_accuracy), torch_test_accuracy]
    }
).set_index("Модель")
display(test_results.round(4))